In [57]:
import pandas as pd

In [58]:
# MBC 지반침하 고위험지역 주소
sinkhole_high_risk = pd.read_csv('./data/processed/지반침하_고위험지역_주소.csv')
# 서울시 건출묵 유출지하수
building_underground_water = pd.read_csv('./data/processed/building_underground_water_processed.csv')
# 서울시 건설 알림이 정보
construction = pd.read_csv('./data/processed/construction_processed.csv')
# 서울시 공사현장 유출지하수
construction_underground_water = pd.read_csv('./data/processed/construction_underground_water_processed.csv')
# 서울시 도로굴착 공사
road_construction = pd.read_csv('./data/processed/road_construction_processed.csv')
# 지하철 깊이
subway_depth = pd.read_csv('./data/processed/subway_depth.csv')

In [59]:
# 이 파일에 피쳐들 저장예정
features = pd.read_csv('./data/processed/subway_depth.csv')

## 지하철역 기본 구조

In [60]:
# 지하철역 기본 구조
import numpy as np
from geopy.distance import geodesic

# 정거장 깊이
features['station_depth'] = subway_depth['지반고'] - subway_depth['레일면고']

# 지하 층 수
features['underground_floors'] = subway_depth['층수'].str.extract('지하(\d+)')
features['underground_floors'] = pd.to_numeric(features['underground_floors'], errors='coerce').fillna(1)

# 플랫폼 형식별 복잡도
features['is_island_platform'] = (subway_depth['형식'] == '섬식').astype(int)
features['is_side_platform'] = (subway_depth['형식'] == '상대식').astype(int)

# 환승역 여부
transfer_keywords = ['환승', '분기', '연결']
features['is_transfer_station'] = subway_depth['비고'].str.contains('|'.join(transfer_keywords), na=False).astype(int)

# 호선별 특성 (건설 시기와 기술 수준)
features['line_1to4'] = subway_depth['호선'].isin([1,2,3,4]).astype(int)
features['line_5to8'] = subway_depth['호선'].isin([5,6,7,8]).astype(int)

<>:9: SyntaxWarning: invalid escape sequence '\d'
<>:9: SyntaxWarning: invalid escape sequence '\d'
/var/folders/mh/1w84fr7s5kxcwc2l24qrjjwc0000gn/T/ipykernel_11916/725895746.py:9: SyntaxWarning: invalid escape sequence '\d'
  features['underground_floors'] = subway_depth['층수'].str.extract('지하(\d+)')


In [61]:
features.head()

,연번,호선,역명,층수,형식,지반고,레일면고,선로기준정거장깊이,정거장깊이,비고,위도,경도,station_depth,underground_floors,is_island_platform,is_side_platform,is_transfer_station,line_1to4,line_5to8
0,1,1,서울,B2,섬식,129.99,117.04,12.95,11.85,"4호선,경의중앙선,공항철도환승",37.553150,126.972533,12.95,1.0,1,0,1,1,0
1,2,1,시청,B2,상대식,129.97,118.82,11.15,10.05,2호선환승,37.563590,126.975407,11.15,1.0,0,1,1,1,0
2,3,1,종각,B2,상대식,128.77,116.24,12.53,11.43,NaN,37.570203,126.983116,12.53,1.0,0,1,0,1,0
3,4,1,종로3가,B2,상대식,124.38,112.04,12.34,11.24,"3,5호선환승",37.570429,126.992095,12.34,1.0,0,1,1,1,0
4,5,1,종로5가,B2,상대식,121.75,109.26,12.49,11.39,NaN,37.570971,127.001900,12.49,1.0,0,1,0,1,0


## 지반침하 위험도

In [62]:
feats = []

for _, station in subway_depth.iterrows():
  station_coords = (station['위도'], station['경도'])
  
  # 거리 계산
  distances = []
  for _, risk_area in sinkhole_high_risk.iterrows():
    if pd.notna(risk_area['lat1']) and pd.notna(risk_area['lon1']):
      risk_coords = (risk_area['lat1'], risk_area['lon1'])
      distance = geodesic(station_coords, risk_coords).meters
      distances.append(distance)
      
  if distances:
    # 반경별 위험지역 개수
    risk_500m = sum(1 for d in distances if d <= 500)
    risk_1km = sum(1 for d in distances if d <= 1000)
    
    # 가장 가까운거
    nearest_distance = min(distances)
    
    # 위험도 가중 (거리 역)
    weighted_score = sum(1000/d for d in distances if d <= 500 and d > 0)
  else:
    risk_500m = risk_1km = 0
    nearest_distance = float('inf')
    weighted_score = 0
    
  feats.append({
    'station_name': station['역명'],
    'risk_area_count_500m': risk_500m,
    'risk_area_count_1km': risk_1km,
    'nearest_risk_distance': nearest_distance,
    'risk_weighted_score_500m': weighted_score
  })

In [63]:
feats = pd.DataFrame(feats)
features.rename(columns={'역명':'station_name'}, inplace=True)
features = features.merge(feats, on='station_name', how='left')

In [64]:
features.head()

,연번,호선,station_name,층수,형식,지반고,레일면고,선로기준정거장깊이,정거장깊이,비고,...,underground_floors,is_island_platform,is_side_platform,is_transfer_station,line_1to4,line_5to8,risk_area_count_500m,risk_area_count_1km,nearest_risk_distance,risk_weighted_score_500m
0,1,1,서울,B2,섬식,129.99,117.04,12.95,11.85,"4호선,경의중앙선,공항철도환승",...,1.0,1,0,1,1,0,0,0,2392.534837,0.000000
1,1,1,서울,B2,섬식,129.99,117.04,12.95,11.85,"4호선,경의중앙선,공항철도환승",...,1.0,1,0,1,1,0,0,0,2376.805396,0.000000
2,2,1,시청,B2,상대식,129.97,118.82,11.15,10.05,2호선환승,...,1.0,0,1,1,1,0,0,0,1322.292309,0.000000
3,2,1,시청,B2,상대식,129.97,118.82,11.15,10.05,2호선환승,...,1.0,0,1,1,1,0,0,0,1335.348492,0.000000
4,3,1,종각,B2,상대식,128.77,116.24,12.53,11.43,NaN,...,1.0,0,1,0,1,0,4,4,333.702328,10.042764


## 건설공사 활동

In [65]:
feats2 = []
for _, station in subway_depth.iterrows():
  station_coords = (station['위도'], station['경도'])
  
  nearby_construction = []
  for _, const in construction.iterrows():
    if pd.notna(const['위치좌표(위도)']) and pd.notna(const['위치좌표(경도)']):
      const_coords = (const['위치좌표(위도)'], const['위치좌표(경도)'])
      distance = geodesic(station_coords, const_coords).meters
      
      if distance <= 500:
        nearby_construction.append({
          'distance': distance,
          'amount': const['사업금액(억원)'] if pd.notna(const['사업금액(억원)']) else 0,
          'is_ongoing': not const['프로젝트 종료여부'],
          
          'is_subway_related': '지하철' in str(const['프로젝트 명']),
          'is_large': const['사업금액(억원)'] > 100 if pd.notna(const['사업금액(억원)']) else False
        })
        
  # 피처 계산
  total_construction = len(nearby_construction)
  ongoing_construction = sum(1 for c in nearby_construction if c['is_ongoing'])
  total_amount = sum(c['amount'] for c in nearby_construction)
  avg_amount = total_amount / len(nearby_construction) if nearby_construction else 0
  subway_related = sum(1 for c in nearby_construction if c['is_subway_related'])
  large_construction = sum(1 for c in nearby_construction if c['is_large'])
  
  feats2.append({
    'station_name': station['역명'],
    'construction_count_500m': total_construction,
    'ongoing_construction_500m': ongoing_construction,
    'total_construction_amount_500m': total_amount,
    'avg_construction_amount_500m': avg_amount,
    'subway_related_construction_500m': subway_related,
    'large_construction_500m': large_construction,
    'construction_density_500m': total_construction / (np.pi * 0.5**2) # 개/km²
  })

In [66]:
feats2 = pd.DataFrame(feats2)
features = features.merge(feats2, on='station_name', how='left')
features.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 522 entries, 0 to 521
Data columns (total 30 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   연번                                522 non-null    int64  
 1   호선                                522 non-null    int64  
 2   station_name                      522 non-null    object 
 3   층수                                522 non-null    object 
 4   형식                                522 non-null    object 
 5   지반고                               522 non-null    float64
 6   레일면고                              522 non-null    float64
 7   선로기준정거장깊이                         522 non-null    float64
 8   정거장깊이                             522 non-null    float64
 9   비고                                352 non-null    object 
 10  위도                                522 non-null    float64
 11  경도                                522 non-null    float64
 12  station_

## 지하수 활동

In [67]:
building_underground_water.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17996 entries, 0 to 17995
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   건축물명            17996 non-null  object 
 1   건축물 위치          17996 non-null  object 
 2   층수(지상_지하)       15420 non-null  object 
 3   조사년도            17996 non-null  int64  
 4   총발생량(톤)         17410 non-null  float64
 5   일평균발생량(톤/일)     17910 non-null  float64
 6   미사용_하수도방류(톤/일)  17733 non-null  float64
 7   lat             15465 non-null  float64
 8   lon             15465 non-null  float64
dtypes: float64(5), int64(1), object(3)
memory usage: 1.2+ MB


In [ ]:
# 먼저 지오코딩

# import asyncio
# import aiohttp

# import nest_asyncio
# nest_asyncio.apply()

# API_KEY = ""
# SEM_LIMIT = 5  # 동시 요청 수

# async def geocode_one(session, addr, sem):
#     clean_addr = clean_address(addr)
#     url = "https://dapi.kakao.com/v2/local/search/address.json"
#     headers = {"Authorization": f"KakaoAK {API_KEY}"}
#     params  = {"query": clean_addr}
#     async with sem:
#         async with session.get(url, headers=headers, params=params, timeout=10) as resp:
#             data = await resp.json()
#     docs = data.get("documents", [])
#     if not docs:
#         return addr, (None, None)
#     y, x = float(docs[0]["y"]), float(docs[0]["x"])
#     return addr, (y, x)

# async def batch_geocode(addrs):
#     sem = asyncio.Semaphore(SEM_LIMIT)
#     async with aiohttp.ClientSession() as session:
#         tasks = [geocode_one(session, addr, sem) for addr in addrs]
#         return await asyncio.gather(*tasks)

# # 실행
# unique_addrs = building_underground_water["건축물 위치"].unique().tolist()
# results = asyncio.run(batch_geocode(unique_addrs))

# # 캐시화
# geocode_cache = dict(results)

# # DF 반영
# building_underground_water["lat"] = building_underground_water["건축물 위치"].map(lambda a: geocode_cache.get(a,(None,None))[0])
# building_underground_water["lon"] = building_underground_water["건축물 위치"].map(lambda a: geocode_cache.get(a,(None,None))[1])

In [ ]:
# # 지오코딩한거 저장
# building_underground_water.to_csv(
#     './data/processed/building_underground_water_processed.csv',
#     index=False,
# )

In [68]:
feats3 = []

for _, station in subway_depth.iterrows():
  station_coords = (station['위도'], station['경도'])
  
  nearby_water = []
  for _, water in building_underground_water.iterrows():
    if pd.notna(water.get('lat')) and pd.notna(water.get('lon')):
      water_coords = (water['lat'], water['lon'])
      distance = geodesic(station_coords, water_coords).meters
      
      if distance <= 500:
        nearby_water.append({
          'distance': distance,
          'daily_output': water['일평균발생량(톤/일)'],
          'waste_output': water['미사용_하수도방류(톤/일)'],
          'floors': water['층수(지상_지하)'],
          'is_large': water['일평균발생량(톤/일)'] > 100
        })
        
  # 피처 계산
  water_building_count = len(nearby_water)
  total_output = sum(w['daily_output'] for w in nearby_water if pd.notna(w['daily_output']))
  total_waste = sum(w['waste_output'] for w in nearby_water if pd.notna(w['waste_output']))
  avg_output = total_output / len(nearby_water) if nearby_water else 0
  large_buildings = sum(1 for w in nearby_water if w['is_large'])
  max_output = max([w['daily_output'] for w in nearby_water if pd.notna(w['daily_output'])], default=0)
  
  feats3.append({
    'station_name': station['역명'],
    'water_building_count_500m': water_building_count,
    'total_water_output_500m': total_output,
    'total_waste_output_500m': total_waste,
    'avg_water_output_500m': avg_output,
    'large_water_buildings_500m': large_buildings,
    'max_water_output_500m': max_output,
    'water_building_density_500m': water_building_count / (np.pi*0.5**2)
  })

In [69]:
feats3 = pd.DataFrame(feats3)
features = features.merge(feats3, on='station_name', how='left')
features.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 894 entries, 0 to 893
Data columns (total 37 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   연번                                894 non-null    int64  
 1   호선                                894 non-null    int64  
 2   station_name                      894 non-null    object 
 3   층수                                894 non-null    object 
 4   형식                                894 non-null    object 
 5   지반고                               894 non-null    float64
 6   레일면고                              894 non-null    float64
 7   선로기준정거장깊이                         894 non-null    float64
 8   정거장깊이                             894 non-null    float64
 9   비고                                720 non-null    object 
 10  위도                                894 non-null    float64
 11  경도                                894 non-null    float64
 12  station_

## 도로굴착공사

In [70]:
# 공사 유형 분류
road_construction['is_subway_related'] = road_construction['공사명'].str.contains('지하철|동북선|신안산선', na=False)
road_construction['is_communication'] = road_construction['신청자'].str.contains('KT|SK|LG|통신', na=False)
road_construction['is_water_sewer'] = road_construction['공사명'].str.contains('상수도|하수|급수', na=False)
road_construction['is_major_road'] = road_construction['도로'] == '특별시도'

# 공시 기간 계산
road_construction['착공일'] = pd.to_datetime(road_construction['착공일'], errors='coerce')
road_construction['준공일'] = pd.to_datetime(road_construction['준공일'], errors='coerce')
road_construction['duration'] = road_construction['공사일수'].fillna(
    (road_construction['준공일'] - road_construction['착공일']).dt.days
)

feats4 = []

for _, station in subway_depth.iterrows():
  station_district = station.get('자치구', '')
  nearby_roads = road_construction[road_construction['구'].str.contains(station_district, na=False)]

  # 피처 계산
  total_road_construction = len(nearby_roads)
  ongoing_road = len(nearby_roads[nearby_roads['처리상태'].str.contains('진행|착공', na=False)])
  subway_related = len(nearby_roads[nearby_roads['is_subway_related']])
  communication_infra = len(nearby_roads[nearby_roads['is_communication']])
  water_sewer = len(nearby_roads[nearby_roads['is_water_sewer']])
  major_road = len(nearby_roads[nearby_roads['is_major_road']])
  avg_duration = nearby_roads['duration'].mean() if len(nearby_roads) > 0 else 0
  long_term = len(nearby_roads[nearby_roads['duration'] > 365])

  feats4.append({
      'station_name': station['역명'],
      'road_construction_count_500m': total_road_construction,
      'ongoing_road_construction_500m': ongoing_road,
      'subway_related_road_500m': subway_related,
      'communication_infra_500m': communication_infra,
      'water_sewer_construction_500m': water_sewer,
      'major_road_construction_500m': major_road,
      'avg_construction_duration_500m': avg_duration,
      'long_term_construction_500m': long_term,
      'road_construction_density_500m': total_road_construction / (np.pi * 0.5**2)
  })

In [71]:
feats4 = pd.DataFrame(feats4)
features = features.merge(feats4, on='station_name', how='left')
features.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1746 entries, 0 to 1745
Data columns (total 46 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   연번                                1746 non-null   int64  
 1   호선                                1746 non-null   int64  
 2   station_name                      1746 non-null   object 
 3   층수                                1746 non-null   object 
 4   형식                                1746 non-null   object 
 5   지반고                               1746 non-null   float64
 6   레일면고                              1746 non-null   float64
 7   선로기준정거장깊이                         1746 non-null   float64
 8   정거장깊이                             1746 non-null   float64
 9   비고                                1564 non-null   object 
 10  위도                                1746 non-null   float64
 11  경도                                1746 non-null   float64
 12  statio

In [72]:
features.to_csv(
  './data/processed/feature_engineering.csv',
  index=False,
  encoding='utf-8'
)

## 종합 위험도

In [73]:
# 종합 위험도 점수 (가중 평균)
features['comprehensive_risk_score'] = (
  features['risk_area_count_500m'] * 0.4 +                    # 지반침하 위험도 40%
  features['ongoing_construction_500m'] * 0.25 +              # 진행중 공사 25%
  features['total_water_output_500m'] / 1000 * 0.2 +          # 지하수 활동 20%
  features['road_construction_count_500m'] * 0.15             # 도로공사 15%
)
    
# 지하 활동 집중도
features['underground_activity_density'] = (
  features['water_building_count_500m'] + 
  features['ongoing_construction_500m'] +
  features['road_construction_count_500m']
)
    
# 지반 교란 지수
features['ground_disturbance_index'] = (
  features['large_construction_500m'] * 2 +
  features['ongoing_road_construction_500m'] +
  features['long_term_construction_500m'] * 1.5
)
    
# 인프라 밀도 지수
features['infrastructure_density'] = (
  features['subway_related_construction_500m'] * 3 +
  features['communication_infra_500m'] +
  features['water_sewer_construction_500m'] * 2
)
    
# 위험도 등급 분류
features['risk_level'] = pd.cut(
  features['comprehensive_risk_score'], 
  bins=4,
  labels=['Low', 'Medium', 'High', 'Very High']
)

In [74]:
features.to_csv(
  './data/processed/feature_engineering.csv',
  index=False,
  encoding='utf-8'
)